In [1]:
!pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 MB 13.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.14-cp311-cp311-linux_x86_64.whl size=4299372 sha256=54e2a6f3fb8cffcc743595d3aeb3a0b015a13953ab880bf1ae310caa0c104a41
  Stored in directory: /root/.cache/pip/wheels/3f/b6/cf/7315ec7b0149210d2d4447d9c3338b36d10e56a1ecddcd35c0
Successfully built llama-cpp-python


In [2]:
!wget https://huggingface.co/MaziyarPanahi/gemma-3-1b-it-GGUF/resolve/main/gemma-3-1b-it.Q8_0.gguf -O gemma-3-1b-it-Q8_0.gguf

--2025-07-30 14:15:01--  https://huggingface.co/MaziyarPanahi/gemma-3-1b-it-GGUF/resolve/main/gemma-3-1b-it.Q8_0.gguf
Resolving huggingface.co (huggingface.co)... 18.164.174.55, 18.164.174.17, 18.164.174.23, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.55|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs-us-1.hf.co/repos/30/98/3098b37681411cb3ef2ad6ed32143545199e6b34e74acbd09d9270e1fccc4e70/b0330a2205ca2c7a243d4a67be42b1f49ac66089d5d318970896f9f7291f020e?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27gemma-3-1b-it.Q8_0.gguf%3B+filename%3D%22gemma-3-1b-it.Q8_0.gguf%22%3B&Expires=1753888147&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc1Mzg4ODE0N319LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zLzMwLzk4LzMwOThiMzc2ODE0MTFjYjNlZjJhZDZlZDMyMTQzNTQ1MTk5ZTZiMzRlNzRhY2JkMDlkOTI3MGUxZmNjYzRlNzAvYjAzMzBhMjIwNWNhMmM3YTI0M2Q0YTY3YmU0MmIxZjQ5YWM2NjA4OWQ1ZDMxODk3MDg5N

In [3]:
!git clone https://github.com/ggerganov/llama.cpp.git

Cloning into 'llama.cpp'...
remote: Enumerating objects: 57760, done.
remote: Counting objects: 100% (239/239), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 57760 (delta 160), reused 65 (delta 64), pack-reused 57521 (from 4)
Receiving objects: 100% (57760/57760), 137.08 MiB | 29.43 MiB/s, done.
Resolving deltas: 100% (41804/41804), done.


In [4]:
from llama_cpp import Llama

# Charger le modèle GGUF
llm = Llama(
    model_path="gemma-3-1b-it-Q8_0.gguf",  # Chemin vers le modèle téléchargé
    n_gpu_layers=35,        # Nombre de couches sur GPU (ajuster selon la mémoire)
    n_threads=4,            # Nombre de threads CPU
    n_ctx=2048,             # Taille du contexte
    verbose=False           # Réduire les messages de debug
)

# Test 1: Génération de texte sur le système solaire
prompt = "Explain how the solar system formed."

output = llm(
    prompt=prompt,
    max_tokens=200,
    temperature=0.7,
    top_p=0.9
)
print("=== Explication du système solaire ===")
print(output["choices"][0]["text"])
print("\n" + "="*50 + "\n")

# Test 2: Génération de code Python avec streaming
print("=== Génération de code Python (streaming) ===")
output_stream = llm(
    prompt="Write a Python script that loads a Hugging Face model and tokenizes input.",
    max_tokens=300,
    temperature=0.3,  # Plus déterministe pour le code
    stream=True       # Activer le streaming
)

# Itérer à travers les tokens et les afficher en temps réel
for output in output_stream:
    token = output["choices"][0]["text"]
    print(token, end="", flush=True)

print("\n" + "="*50 + "\n")

# Fonction utilitaire pour tester différents prompts
def generate_code(prompt, max_tokens=250):
    """Fonction helper pour générer du code avec des paramètres optimisés"""
    response = llm(
        prompt=f"# Python code:\n{prompt}\n\n",
        max_tokens=max_tokens,
        temperature=0.2,  # Très déterministe pour le code
        top_p=0.95,
        stop=["```", "#", "\n\n\n"]  # Arrêter à certains délimiteurs
    )
    return response["choices"][0]["text"].strip()

# Test avec la fonction helper
print("=== Test avec fonction helper ===")
code_prompt = "Create a function that checks if a number is prime"
generated_code = generate_code(code_prompt)
print(generated_code)

llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache_unified: LLAMA_SET_ROWS=0, using old ggml_cpy() method for backwards compatibility
llama_kv_cache_unified: LLAMA_SET_ROWS=0, using old ggml_cpy() method for backwards compatibility


=== Explication du système solaire ===


Okay, let's break down how the solar system formed. It's a fascinating story involving gravity, gas, and time. Here's a simplified explanation:

**1. The Beginning: A Giant Molecular Cloud**

* **The Starting Point:** About 4.6 billion years ago, there was a massive, cold cloud of gas and dust called a molecular cloud. These clouds were incredibly large – much bigger than our solar system!
* **Turbulence and Collapse:** Within the cloud, pockets of gas started to collapse under their own gravity.  This collapse was triggered by a shockwave, likely caused by a nearby supernova explosion.  As the cloud collapsed, it began to spin faster and faster.

**2. The Birth of the Protosun**

* **Gravitational Heating:** As the cloud collapsed, it flattened into a rotating disk.  The spinning caused the cloud's material to heat up due to friction.
* **Nuclear Fusion Ignition


=== Génération de code Python (streaming) ===


```python
from transformers impor

In [9]:
print(generate_code("Create a function that checks if a number is prime"))

def is_prime(n):
  """
  Checks if a number is prime.
  
  Args:
    n: An integer.
  
  Returns:
    True if n is prime, False otherwise.
  """
  if n <= 1:
    return False
  for i in range(2, int(n**0.5) + 1):
    if n % i == 0:
      return False
  return True


In [14]:
print("=== Génération CSV + Matplotlib (streaming) ===")
output_stream = llm(
    prompt="Write a complete Python script that reads data.csv and creates a line plot:",
    max_tokens=400,
    temperature=0.3,
    stream=True
)

for output in output_stream:
    token = output["choices"][0]["text"]
    print(token, end="", flush=True)

=== Génération CSV + Matplotlib (streaming) ===


```
import pandas as pd
import matplotlib.pyplot as plt

# Read the data
try:
    df = pd.read_csv('data.csv')
except FileNotFoundError:
    print("Error: data.csv not found.")
    exit()
except Exception as e:
    print(f"An error occurred: {e}")
    exit()

# Create the plot
plt.plot(df['column_name'])

# Add labels and title
plt.xlabel("Column Name")
plt.ylabel("Column Value")
plt.title("Line Plot of Data")

# Show the plot
plt.show()
```

**Explanation:**

1.  **Import Libraries:**
    *   `pandas` is used for data manipulation and analysis.
    *   `matplotlib.pyplot` is used for creating plots.

2.  **Read Data:**
    *   `pd.read_csv('data.csv')` reads the data from the file named 'data.csv' into a pandas DataFrame.
    *   The `try...except` block handles potential errors like the file not being found.

3.  **Create the Plot:**
    *   `plt.plot(df['column_name'])` creates a line plot using the 'column\_name' column from the Dat

In [15]:
print("=== Web Scraper ===")
output_stream = llm(
    prompt="Create a simple web scraper that extracts headlines using requests and BeautifulSoup",
    max_tokens=500,
    temperature=0.2,
    stream=True
)

for output in output_stream:
    token = output["choices"][0]["text"]
    print(token, end="", flush=True)

=== Web Scraper ===
.

```python
import requests
from bs4 import BeautifulSoup

def scrape_headlines(url):
    """
    Scrapes headlines from a given URL.

    Args:
        url: The URL to scrape.

    Returns:
        A list of headlines.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for bad status codes

        soup = BeautifulSoup(response.content, 'html.parser')
        headlines = []
        for h2 in soup.find_all('h2'):
            headlines.append(h2.text.strip())
        return headlines
    except requests.exceptions.RequestException as e:
        print(f"Error during request: {e}")
        return []
    except Exception as e:
        print(f"An error occurred: {e}")
        return []

if __name__ == '__main__':
    url = "https://www.example.com"  # Replace with the URL you want to scrape
    headlines = scrape_headlines(url)
    if headlines:
        print("Headlines:")
        for headline in headlines:
  

In [16]:
print("=== List vs Tuples ===")
output_stream = llm(
    prompt="Explain the difference between a list and a tuple in Python",
    max_tokens=500,
    temperature=0.9,
    stream=True
)

for output in output_stream:
    token = output["choices"][0]["text"]
    print(token, end="", flush=True)

=== List vs Tuples ===
?

**List vs. Tuple in Python**

Both lists and tuples are used to store collections of data in Python. However, there are some key differences that make them suitable for different situations.

1. **Syntax:**
   - **List:** Defined using square brackets `[]`
   - **Tuple:** Defined using parentheses `()`

2. **Mutability:**
   - **List:** Mutable – you can change the elements of a list after it's created.
   - **Tuple:** Immutable – once a tuple is created, its elements cannot be changed.

3. **Performance:**
   - **List:**  Generally slower than tuples, especially when performing operations that modify the list.
   - **Tuple:** Generally faster than lists.

4. **Use Cases:**
   - **List:** Suitable for situations where you need to modify the data, such as storing a dynamic list of items.
   - **Tuple:** Suitable for situations where you need to ensure data integrity, such as representing a fixed set of data.

**Example:**

```python
# List example
my_list = [1,

In [17]:
print("=== List vs Tuples ===")
output_stream = llm(
    prompt="Explain the difference between a list and a tuple in Python",
    max_tokens=500,
    temperature=0.1,
    stream=True
)

for output in output_stream:
    token = output["choices"][0]["text"]
    print(token, end="", flush=True)

=== List vs Tuples ===
?

**Answer:**

In Python, both lists and tuples are used to store collections of items, but they have key differences that affect how they are used and how they behave.

*   **Lists:**
    *   **Mutable:** Lists are mutable, meaning you can change their contents after they are created. You can add, remove, or modify elements.
    *   **Ordered:** Lists are ordered, meaning the elements are stored in a specific sequence.
    *   **Dynamic Size:** Lists can grow or shrink in size as needed.

*   **Tuples:**
    *   **Immutable:** Tuples are immutable, meaning you cannot change their contents after they are created. You cannot add, remove, or modify elements.
    *   **Indexed:** Tuples are indexed, meaning you can access elements by their position (index).
    *   **Fixed Size:** Tuples have a fixed size, which is determined at the time of creation.

**Here's a table summarizing the key differences:**

| Feature          | List                     | Tuple         

In [18]:
print("=== Even Loop ===")
output_stream = llm(
    prompt="Generate a for-loop that prints even numbers between 1 and 100",
    max_tokens=500,
    temperature=0.5,
    stream=True
)

for output in output_stream:
    token = output["choices"][0]["text"]
    print(token, end="", flush=True)

=== Even Loop ===
.

```python
for i in range(2, 101, 2):
    print(i)
```

**Explanation:**

*   `range(2, 101, 2)`: This creates a sequence of numbers starting from 2, up to (but not including) 101, with a step of 2.  So, it generates the numbers 2, 4, 6, ..., 100.
*   `for i in ...`: The `for` loop iterates through each number generated by `range()`.
*   `print(i)`: Inside the loop, the current number `i` is printed to the console.

This code will print all even numbers between 1 and 100.
